In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("/Users/marktatrai/Python_feladatok/supermarket/supermarket_large_dataset.csv")
df.head()

,transaction_id,transaction_line_id,basket_id,customer_id,store_id,product_id,employee_id,supplier_id,customer_age,customer_gender,...,units_sold,sales_revenue,profit_per_unit,profit_margin,sales_rank,category_sales_rank,product_popularity,product_growth_rate,product_return_rate,product_stockout_rate
0,T00000001,TL00000001,B00044698,C00106069,S00000108,P00003814,E00000421,SUP00000007,60,Female,...,10,100.60,4.65,0.462,883,10,5.6,0.064,0.0580,0.0293
1,T00000002,TL00000002,B00013390,C00027234,S00000154,P00003904,E00000310,SUP00000035,54,Male,...,6,96.90,8.66,0.536,599,29,6.0,-0.025,0.0071,0.0028
2,T00000003,TL00000003,B00014357,C00019781,S00000136,P00000650,E00000344,SUP00000022,52,Male,...,6,132.18,7.90,0.359,389,49,5.2,-0.038,0.0029,0.0553
3,T00000004,TL00000004,B00046655,C00018191,S00000169,P00000966,E00000168,SUP00000005,40,Male,...,12,54.36,1.97,0.435,111,70,3.8,0.189,0.0146,0.0526
4,T00000005,TL00000005,B00046461,C00031493,S00000179,P00000584,E00000455,SUP00000014,44,Female,...,5,21.75,0.04,0.009,794,7,5.8,0.116,0.0479,0.1025


In [22]:
df.shape

(120000, 323)

In [23]:
df.dtypes.value_counts()

str        143
float64    103
int64       77
Name: count, dtype: int64

In [24]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(100,100))
sns.heatmap(df.corr(numeric_only=True))

<Axes: >

In [ ]:
# observing  higher correlation connections
corr_inv = df[['inventory_value','discount_amount','discount_percent', 'profit_margin']].corr()
corr_inv

# also the tax rate and the profit margin also has significant correlation

,inventory_value,discount_amount,discount_percent,profit_margin
inventory_value,1.000000,0.355012,-0.000580,-0.017711
discount_amount,0.355012,1.000000,0.509688,-0.280490
discount_percent,-0.000580,0.509688,1.000000,-0.683175
profit_margin,-0.017711,-0.280490,-0.683175,1.000000


#### significant correlations: tax amount and profit per unit (0.516)

In [57]:
corr_tax = df[['tax_rate', 'tax_amount', 'profit_margin', 'profit_per_unit']].corr()
corr_tax

,tax_rate,tax_amount,profit_margin,profit_per_unit
tax_rate,1.000000,0.451537,0.000263,0.005532
tax_amount,0.451537,1.000000,0.133792,0.515741
profit_margin,0.000263,0.133792,1.000000,0.506160
profit_per_unit,0.005532,0.515741,0.506160,1.000000


In [25]:
df_float=df.select_dtypes(include='float').isnull().sum().sort_values(ascending=False)
df_float


complaint_resolution_time_hours    7285
delivery_time_minutes              6016
expiry_risk_score                  5986
marketing_engagement_score         4785
demand_volatility                  4772
                                   ... 
product_popularity_score              0
basket_value                          0
average_discount_percent              0
average_item_price                    0
product_stockout_rate                 0
Length: 103, dtype: int64

In [49]:
nuniq = df.select_dtypes(include=['object','str']).nunique()
nuniq.sort_values(ascending=False)

transaction_id          120000
transaction_line_id     120000
transaction_datetime    119798
customer_id              75799
transaction_time         50521
                         ...  
repeat_purchase              2
country                      1
reorder_status               1
low_stock_status             1
stockout_status              1
Length: 143, dtype: int64

In [27]:
ohc_cols = nuniq[(nuniq>1) & (nuniq<13)].index.tolist()
print(f'{len(ohc_cols)} columns will be encoded')

118 columns will be encoded


In [28]:
ohc_cols_encoded = pd.get_dummies(df, columns=ohc_cols, drop_first=True)
print('from', df.shape, '->', ohc_cols_encoded.shape, 'was created')

from (120000, 323) -> (120000, 507) was created


In [29]:
ohc_cols_encoded.dtypes.value_counts()

bool       302
float64    103
int64       77
str         25
Name: count, dtype: int64

In [30]:
print(ohc_cols_encoded['customer_income'].value_counts())
print(ohc_cols_encoded['customer_income'].mean())

#we can see, that the customer income can be dropped as its values (which are not missing) are all 250000, with no variance

customer_income
250000.0    117529
Name: count, dtype: int64
250000.0


In [31]:
ohc_cols_encoded.describe()

#we can see that several variables have 0 std, so let's filter those

,customer_age,customer_income,household_size,number_of_children,customer_since_year,customer_tenure_months,loyalty_points_earned,loyalty_points_redeemed,loyalty_points_balance,membership_duration_months,...,units_sold,sales_revenue,profit_per_unit,profit_margin,sales_rank,category_sales_rank,product_popularity,product_growth_rate,product_return_rate,product_stockout_rate
count,120000.000000,117529.0,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,...,120000.000000,120000.000000,120000.000000,120000.00000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000,120000.000000
mean,44.787608,250000.0,3.002558,0.846308,2016.503833,95.460967,49.950825,12.456842,287.743925,95.460967,...,8.509900,83.297038,3.696514,0.33056,499.706358,50.528283,4.996500,0.050269,0.030626,0.050599
std,14.435695,0.0,1.378411,0.860916,4.023941,48.442105,50.052337,16.156370,149.728718,48.442105,...,4.341796,98.899303,4.678706,0.20422,288.329386,28.833530,1.974317,0.149760,0.018858,0.028797
min,18.000000,250000.0,1.000000,0.000000,2010.000000,12.000000,0.000000,0.000000,0.000000,12.000000,...,1.000000,0.520000,-8.920000,-0.65900,1.000000,1.000000,0.000000,-0.500000,0.000000,0.000000
25%,35.000000,250000.0,2.000000,0.000000,2013.000000,53.000000,14.000000,2.000000,162.000000,53.000000,...,5.000000,26.400000,0.960000,0.22700,250.000000,25.000000,3.600000,-0.051000,0.016600,0.029700
50%,45.000000,250000.0,3.000000,1.000000,2016.000000,96.000000,35.000000,7.000000,288.000000,96.000000,...,8.000000,52.950000,2.300000,0.37300,499.000000,51.000000,5.000000,0.051000,0.030000,0.049900
75%,55.000000,250000.0,4.000000,1.000000,2020.000000,137.000000,69.000000,16.000000,413.000000,137.000000,...,11.000000,102.360000,4.890000,0.48000,750.000000,75.000000,6.300000,0.152000,0.043500,0.070300
max,85.000000,250000.0,6.000000,5.000000,2023.000000,179.000000,581.000000,219.000000,836.000000,179.000000,...,40.000000,1888.200000,62.120000,0.60000,1000.000000,100.000000,10.000000,0.724000,0.115400,0.187800


In [32]:
# let's see which variables can be dropped, as they do not contribute to the assessment
zero_std = ohc_cols_encoded.std(numeric_only=True)
zero_std=zero_std[zero_std==0].index.tolist()

In [33]:
ohc_cols_encoded=ohc_cols_encoded.drop(columns=zero_std)

### Before the correlation matrix we should standardize the numeric variables!!

In [34]:
corr = ohc_cols_encoded.corr(numeric_only=True)
corr

,customer_age,household_size,number_of_children,customer_since_year,customer_tenure_months,loyalty_points_earned,loyalty_points_redeemed,loyalty_points_balance,membership_duration_months,customer_visit_frequency,...,employee_role_Cashier,employee_role_Customer Service,employee_role_Manager,employee_role_Stock Clerk,employee_shift_Evening,employee_shift_Morning,employee_shift_Night,economic_condition_Stable,economic_condition_Strong,economic_condition_Weak
customer_age,1.000000,0.003069,0.000914,-0.002246,0.001622,-0.002240,0.001859,-0.002839,0.001622,-0.000257,...,0.002452,-0.002327,-0.002272,-0.001101,-0.000678,-0.001550,0.000374,-0.002305,-0.000769,0.000322
household_size,0.003069,1.000000,0.398148,0.003578,-0.003398,-0.000416,-0.001812,0.003075,-0.003398,0.002936,...,-0.002995,-0.002932,0.000733,0.003747,0.004001,0.000741,0.002136,-0.004623,0.001964,0.003794
number_of_children,0.000914,0.398148,1.000000,0.001639,-0.001365,0.004359,0.000844,-0.000906,-0.001365,0.004448,...,-0.005458,0.004988,-0.003376,0.002096,0.003829,-0.003820,0.001310,0.001344,0.000042,0.000618
customer_since_year,-0.002246,0.003578,0.001639,1.000000,-0.997468,0.001737,0.001543,-0.000034,-0.997468,0.000103,...,-0.000548,0.003963,-0.001171,0.000158,0.003123,-0.005984,0.001460,-0.000050,0.000374,-0.000439
customer_tenure_months,0.001622,-0.003398,-0.001365,-0.997468,1.000000,-0.001986,-0.001608,-0.000555,1.000000,0.000107,...,0.000581,-0.003803,0.001193,-0.000459,-0.002969,0.006122,-0.001459,-0.000032,-0.000661,0.000458
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
employee_shift_Morning,-0.001550,0.000741,-0.003820,-0.005984,0.006122,0.003106,0.002637,0.003019,0.006122,-0.003927,...,-0.004915,-0.001283,0.004336,0.003824,-0.424406,1.000000,-0.246820,0.002960,-0.005717,0.000211
employee_shift_Night,0.000374,0.002136,0.001310,0.001460,-0.001459,-0.003120,-0.000983,0.001651,-0.001459,-0.001361,...,0.001938,-0.001413,-0.001692,0.001961,-0.191905,-0.246820,1.000000,0.002093,-0.002771,0.001454
economic_condition_Stable,-0.002305,-0.004623,0.001344,-0.000050,-0.000032,0.001543,0.000633,0.002219,-0.000032,-0.001746,...,0.001248,-0.001071,0.003241,0.000398,-0.006255,0.002960,0.002093,1.000000,-0.408705,-0.340442
economic_condition_Strong,-0.000769,0.001964,0.000042,0.000374,-0.000661,-0.004411,-0.002724,-0.004077,-0.000661,-0.000482,...,0.001116,-0.001313,0.000182,-0.003398,0.004123,-0.005717,-0.002771,-0.408705,1.000000,-0.209809


In [35]:
# see if there is any missing value
corr.isna().any().any()

np.False_

In [36]:
marital=[col for col in ohc_cols_encoded if 'marital' in col]
marital

['customer_marital_status_Married',
 'customer_marital_status_Single',
 'customer_marital_status_Widowed']

In [37]:
other_cols = [cols for cols in corr.columns if cols not in marital]
marital_corr = corr.loc[marital,other_cols]
marital_corr.T

,customer_marital_status_Married,customer_marital_status_Single,customer_marital_status_Widowed
customer_age,0.000541,0.003548,0.000150
household_size,-0.000055,0.000258,-0.002977
number_of_children,0.000748,-0.001467,-0.001578
customer_since_year,-0.002272,-0.002345,0.002926
customer_tenure_months,0.002225,0.002192,-0.002498
...,...,...,...
employee_shift_Morning,-0.000978,-0.001871,0.000131
employee_shift_Night,0.001125,0.003031,-0.002239
economic_condition_Stable,0.002986,-0.001526,-0.002049
economic_condition_Strong,-0.003552,0.001760,0.003280


In [38]:
income= [cols for cols in ohc_cols_encoded if 'income' in cols]
income

['average_household_income', 'local_average_income']

In [39]:
income_marital_corr = ohc_cols_encoded[marital+income].corr().loc[marital,income]
income_marital_corr

,average_household_income,local_average_income
customer_marital_status_Married,-0.002804,0.002797
customer_marital_status_Single,0.000370,-0.000374
customer_marital_status_Widowed,0.002554,0.000842


In [40]:
geography = [cols for cols in ohc_cols_encoded if 'region' or 'city' or 'urban' or 'suburban' or 'rural' in cols]
geography

['transaction_id',
 'transaction_line_id',
 'basket_id',
 'customer_id',
 'store_id',
 'product_id',
 'employee_id',
 'supplier_id',
 'customer_age',
 'household_size',
 'number_of_children',
 'customer_since_year',
 'customer_tenure_months',
 'loyalty_points_earned',
 'loyalty_points_redeemed',
 'loyalty_points_balance',
 'membership_duration_months',
 'customer_visit_frequency',
 'average_monthly_visits',
 'average_transaction_value',
 'customer_lifetime_value',
 'customer_retention_score',
 'customer_churn_risk',
 'customer_engagement_score',
 'store_age_years',
 'number_of_employees',
 'number_of_checkout_counters',
 'number_of_self_checkout_counters',
 'parking_spaces',
 'daily_customer_capacity',
 'store_rating',
 'country',
 'latitude',
 'longitude',
 'population_density',
 'average_household_income',
 'nearby_competitor_count',
 'product_name',
 'product_category',
 'product_subcategory',
 'brand',
 'package_quantity',
 'package_weight',
 'product_age_days',
 'unit_cost',
 'reg

In [41]:
import numpy as np

corr = ohc_cols_encoded.corr(numeric_only=True)
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))

pairs = (
    upper.stack()
    .rename("korreláció")
    .to_frame()
    .assign(abszolút=lambda x: x["korreláció"].abs())
    .sort_values("abszolút", ascending=False)
)

pairs.head(20)

korreláció  \
remaining_shelf_life_days      days_to_expiry                      1.0   
stock_after_transaction        store_stock                         1.0   
daily_demand                   monthly_demand                      1.0   
                               weekly_demand                       1.0   
digital_wallet_Yes             mobile_payment_Yes                  1.0   
marketing_channel_Mobile App   mobile_app_notification_Yes         1.0   
sales_channel_Home Delivery    home_delivery_Yes                   1.0   
sales_channel_In Store         in_store_purchase_Yes               1.0   
sales_channel_Mobile App       mobile_app_order_Yes                1.0   
store_city_City D              city_City D                         1.0   
sales_channel_Website          website_order_Yes                   1.0   
payment_channel_Mobile App     mobile_payment_Yes                  1.0   
promotion_active_Yes           promotion_type_No Promotion        -1.0   
marketing_channel_SMS          sms_campaign_Yes                    1.0   
quantity_sold                  units_sold                          1.0   
nearby_competitor_count        competitor_count                    1.0   
region_West                    state_province_West                 1.0   
marketing_channel_Social Media social_media_exposure_Yes           1.0   
region_South                   state_province_South                1.0   
region_North                   state_province_North                1.0   

                                                            abszolút  
remaining_shelf_life_days      days_to_expiry                    1.0  
stock_after_transaction        store_stock                       1.0  
daily_demand                   monthly_demand                    1.0  
                               weekly_demand                     1.0  
digital_wallet_Yes             mobile_payment_Yes                1.0  
marketing_channel_Mobile App   mobile_app_notification_Yes       1.0  
sales_channel_Home Delivery    home_delivery_Yes                 1.0  
sales_channel_In Store         in_store_purchase_Yes             1.0  
sales_channel_Mobile App       mobile_app_order_Yes              1.0  
store_city_City D              city_City D                       1.0  
sales_channel_Website          website_order_Yes                 1.0  
payment_channel_Mobile App     mobile_payment_Yes                1.0  
promotion_active_Yes           promotion_type_No Promotion       1.0  
marketing_channel_SMS          sms_campaign_Yes                  1.0  
quantity_sold                  units_sold                        1.0  
nearby_competitor_count        competitor_count                  1.0  
region_West                    state_province_West               1.0  
marketing_channel_Social Media social_media_exposure_Yes         1.0  
region_South                   state_province_South              1.0  
region_North                   state_province_North              1.0

In [42]:
nt_1_pairs = pairs[(pairs['abszolút']<0.9) & (pairs['abszolút']>0.3)].sort_values('abszolút',ascending=False)

In [43]:
nt_1_pairs.head(70)

korreláció  abszolút
delivery_time_minutes sales_channel_Home Delivery    0.897290  0.897290
                      home_delivery_Yes              0.897290  0.897290
gross_profit          sales_revenue                  0.897267  0.897267
selling_price         profit_per_unit                0.884386  0.884386
total_amount          sales_revenue                  0.881868  0.881868
...                                                       ...       ...
storage_temperature   product_perishable_Yes        -0.786970  0.786970
                      perishable_flag_Yes           -0.786970  0.786970
competitor_price      relative_price_index           0.785387  0.785387
regular_price         sales_revenue                  0.781030  0.781030
price_index           sales_revenue                  0.781029  0.781029

[70 rows x 2 columns]

In [44]:
nt_1_pairs.count

<bound method DataFrame.count of                                                            korreláció  \
delivery_time_minutes      sales_channel_Home Delivery       0.897290   
                           home_delivery_Yes                 0.897290   
gross_profit               sales_revenue                     0.897267   
selling_price              profit_per_unit                   0.884386   
total_amount               sales_revenue                     0.881868   
...                                                               ...   
month                      month_name_October                0.307642   
temperature_c              weather_condition_Cold           -0.304636   
payment_method_Credit Card card_type_Visa                    0.302972   
card_type_Visa             credit_card_payment_Yes           0.302972   
return_reason_Expired      refund_method_Original Payment    0.301117   

                                                           abszolút  
delivery_time_minute

In [45]:
df_nt_1_pairs=pd.DataFrame(nt_1_pairs)

In [46]:
df_nt_1_pairs

korreláció  \
delivery_time_minutes      sales_channel_Home Delivery       0.897290   
                           home_delivery_Yes                 0.897290   
gross_profit               sales_revenue                     0.897267   
selling_price              profit_per_unit                   0.884386   
total_amount               sales_revenue                     0.881868   
...                                                               ...   
month                      month_name_October                0.307642   
temperature_c              weather_condition_Cold           -0.304636   
payment_method_Credit Card card_type_Visa                    0.302972   
card_type_Visa             credit_card_payment_Yes           0.302972   
return_reason_Expired      refund_method_Original Payment    0.301117   

                                                           abszolút  
delivery_time_minutes      sales_channel_Home Delivery     0.897290  
                           home_delivery_Yes               0.897290  
gross_profit               sales_revenue                   0.897267  
selling_price              profit_per_unit                 0.884386  
total_amount               sales_revenue                   0.881868  
...                                                             ...  
month                      month_name_October              0.307642  
temperature_c              weather_condition_Cold          0.304636  
payment_method_Credit Card card_type_Visa                  0.302972  
card_type_Visa             credit_card_payment_Yes         0.302972  
return_reason_Expired      refund_method_Original Payment  0.301117  

[623 rows x 2 columns]

In [47]:
nt_1_pairs['abszolút'].plot(kind='barh', figsize=(100,200))
plt.ylabel('absolute correlation')
plt.xlabel('variable pairs')
plt.tight_layout()
plt.show()